# Fine-tuning LLM untuk Chatbot Hukum Indonesia

**Modul**: Pengembangan Generative AI berbasis LLM — Digdaya Hackathon BI

**Target**: Advanced (4 pts)

---

## Ringkasan Notebook

1. Load dataset `Ichsan2895/alpaca-gpt4-indonesian` (format Alpaca, Bahasa Indonesia)
2. Train/validation split (90/10)
3. Mapping dataset ke ChatML template (format native Qwen2.5)
4. Load `Qwen/Qwen2.5-7B-Instruct` dengan QLoRA 4-bit + double quantization
5. LoRA adapters pada komponen Multi-Head Attention
6. Dua eksperimen SFTTrainer dengan hyperparameter berbeda
7. Loss curve comparison
8. Merge LoRA (merged_16bit) + push ke HuggingFace Hub

> **Catatan**: Notebook ini didesain untuk Google Colab dengan GPU T4 (16GB VRAM).

## 1. Instalasi Dependencies

In [ ]:
# Install Unsloth dan dependencies (Colab/Kaggle compatible)
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub safetensors matplotlib

In [ ]:
# [PENTING] Setelah cell instalasi di atas selesai:
# RESTART RUNTIME secara manual: Runtime → Restart runtime (Ctrl+M .)
# Setelah restart, SKIP 2 cell instalasi ini dan LANJUTKAN dari "2. Import Libraries"
# Cell ini hanya instruksi — tidak dijalankan setelah restart
print("\n" + "=" * 60)
print("INSTALASI SELESAI.")
print("SEKARANG: Runtime → Restart runtime (Ctrl+M .)")
print("Setelah restart, skip cell instalasi, lanjut ke IMPORT.")
print("=" * 60)

## 2. Import Libraries

In [ ]:
import unsloth  # noqa: F401 — init Unsloth optimizations (harus paling atas sebelum torch/transformers)

import os
import gc
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from transformers import TrainingArguments
from trl import SFTTrainer
import matplotlib.pyplot as plt

# Cek GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: GPU tidak terdeteksi! Pastikan Runtime → Change runtime type → T4 GPU")

def is_bfloat16_supported():
    """Cek apakah GPU support bfloat16 (T4 tidak support)."""
    if torch.cuda.is_available():
        return torch.cuda.get_device_capability()[0] >= 8
    return False

print(f"BF16 support: {is_bfloat16_supported()}")
print("Import libraries selesai.")

## 3. Load Dataset

Dataset: `Ichsan2895/alpaca-gpt4-indonesian` — format Alpaca (instruction, input, output) dalam Bahasa Indonesia.

In [ ]:
# Load dataset dari HuggingFace
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print(f"Jumlah total sample: {len(dataset)}")
print(f"Kolom: {dataset.column_names}")

In [ ]:
# Tampilkan beberapa baris dataset MENTAH (sebelum mapping)
# Dataset Ichsan2895/alpaca-gpt4-indonesian kolom: ['Unnamed: 0', 'input', 'output']
# - Unnamed: 0 = row index (abaikan)
# - input = instruction + context (gabungan)
# - output = jawaban

print("=" * 80)
print("DATASET MENTAH — Sebelum Chat Template Mapping")
print(f"Kolom: {dataset.column_names}")
print("=" * 80)
for i in range(3):
    sample = dataset[i]
    print(f"--- Sample #{i+1} ---")
    print(f"instruction (input): {str(sample['input'])[:300]}...")
    print(f"output: {str(sample['output'])[:300]}...")
    print()

## 4. Train/Validation Split

Split dataset 90% training, 10% validation untuk memenuhi kriteria Skilled & Advanced.

In [ ]:
# Train/validation split (90/10)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

## 5. Chat Template Mapping

Mapping dataset ke **ChatML format** (format native Qwen2.5).

Format ChatML:
```
<|im_start|>system
{system_message}<|im_end|>
<|im_start|>user
{instruction + input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>
```

Token spesial ChatML:
- `<|im_start|>` — penanda awal pesan (role)
- `<|im_end|>` — penanda akhir pesan
- Role tokens: `system`, `user`, `assistant`

In [ ]:
# Define system prompt dalam Bahasa Indonesia
SYSTEM_PROMPT = """Anda adalah asisten AI yang membantu menjawab pertanyaan dalam Bahasa Indonesia. Berikan jawaban yang akurat, informatif, dan sopan."""

def format_chatml(example):
    """
    Mapping function: mengubah row dataset Alpaca ke format ChatML.
    
    Dataset Ichsan2895/alpaca-gpt4-indonesian kolom:
    - input: berisi instruction/pertanyaan
    - output: berisi jawaban
    
    Format ChatML:
    <|im_start|>system
    {system}<|im_end|>
    <|im_start|>user
    {input}<|im_end|>
    <|im_start|>assistant
    {output}<|im_end|>
    """
    # input = instruction dalam dataset ini
    user_message = example.get("input", "") or ""
    assistant_message = example.get("output", "") or ""
    
    # Bangun teks lengkap format ChatML
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n{assistant_message}<|im_end|>"
    )
    
    return {"text": text}

# Terapkan mapping ke train dan eval dataset
train_dataset_mapped = train_dataset.map(format_chatml)
eval_dataset_mapped = eval_dataset.map(format_chatml)

print("Mapping selesai. Kolom baru: 'text' ditambahkan.")

In [ ]:
# Tampilkan 1 baris dataset yang SUDAH di-mapping (lengkap dengan token spesial)
print("=" * 80)
print("DATASET SUDAH DI-MAPPING — Format ChatML dengan Token Spesial")
print("=" * 80)
print()

sample_mapped = train_dataset_mapped[0]
print(sample_mapped["text"])

print()
print("=" * 80)
print("Token spesial yang digunakan:")
print("  <|im_start|>  — penanda awal role (system/user/assistant)")
print("  <|im_end|>    — penanda akhir pesan")
print("  system, user, assistant — nama role")
print("=" * 80)

## 6. Load Model dengan QLoRA 4-bit + Double Quantization

Menggunakan `Qwen/Qwen2.5-7B-Instruct` sebagai base model. QLoRA configuration:
- **4-bit quantization** dengan `bitsandbytes` (`load_in_4bit=True`)
- **Double quantization** (`bnb_4bit_use_double_quant=True`) — quantisasi ulang konstanta quantisasi untuk mengurangi memory footprint lebih lanjut
- **NF4 datatype** (`bnb_4bit_quant_type="nf4"`) — NormalFloat4, format optimal untuk weight terdistribusi normal

In [ ]:
# Model dan tokenizer
model_name = "Qwen/Qwen2.5-7B-Instruct"
max_seq_length = 1024  # T4-friendly (turun dari 2048)

# Load model dengan QLoRA 4-bit + double quantization via Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print(f"Model loaded: {model_name}")
print(f"Quantization: 4-bit NF4 + Double Quantization (QLoRA)")
print(f"Max sequence length: {max_seq_length}")

In [ ]:
# Verifikasi konfigurasi quantisasi
import bitsandbytes as bnb

quant_config = model.config.quantization_config
if quant_config:
    print("Quantization Config:")
    for key, value in quant_config.items():
        print(f"  {key}: {value}")
else:
    # Unsloth mungkin menyimpan config di tempat berbeda
    print("Quantization dikelola oleh Unsloth secara internal")

## 7. LoRA Configuration

LoRA adapters didefinisikan pada **seluruh komponen Multi-Head Attention (MHA)**:
- `q_proj` — Query projection
- `k_proj` — Key projection
- `v_proj` — Value projection
- `o_proj` — Output projection

Ini memastikan LoRA mencakup setidaknya satu komponen komputasi utama (MHA) secara penuh.

In [ ]:
# Definisikan LoRA adapters pada komponen Multi-Head Attention
# Parameter lora_r dan lora_alpha akan divariasikan per eksperimen

def apply_lora(model, r=16, lora_alpha=16, lora_dropout=0.0):
    """
    Terapkan LoRA adapters pada model.
    
    Target modules: q_proj, k_proj, v_proj, o_proj (seluruh MHA)
    Juga menyertakan gate_proj, up_proj, down_proj (FFN) untuk coverage lebih luas
    """
    model = FastLanguageModel.get_peft_model(
        model,
        r=r,  # LoRA rank
        lora_alpha=lora_alpha,  # Scaling factor
        lora_dropout=lora_dropout,  # Regularization
        target_modules=[
            # Multi-Head Attention (KOMPONEN UTAMA - wajib)
            "q_proj", "k_proj", "v_proj", "o_proj",
            # Feed-Forward Network (tambahan untuk coverage lebih baik)
            "gate_proj", "up_proj", "down_proj",
        ],
        use_gradient_checkpointing="unsloth",  # Memory-efficient
        random_state=42,
        use_rslora=False,  # Standard LoRA
        loftq_config=None,
    )
    return model

print("Fungsi apply_lora() siap digunakan.")
print("Target modules MHA: q_proj, k_proj, v_proj, o_proj")
print("Target modules FFN: gate_proj, up_proj, down_proj")

In [ ]:
# Update tokenizer untuk ChatML
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # Format ChatML untuk Qwen2.5
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

print("Tokenizer updated dengan ChatML template.")

## 8. Eksperimen 1 — Default Hyperparameters (T4 Optimized)

| Parameter | Value |
|-----------|-------|
| learning_rate | 2e-4 |
| per_device_train_batch_size | 1 |
| lora_r | 16 |
| lora_alpha | 16 |
| gradient_accumulation_steps | 8 |
| max_steps | 800 |
| max_seq_length | 1024 |
| eval_strategy | steps |
| eval_steps | 100 |
| logging_steps | 50 |
| lr_scheduler_type | cosine |

> **Catatan OOM T4**: Batch size diturunkan ke 1, seq_length ke 1024, fp16=True untuk menghindari OOM di T4 16GB.

In [ ]:
# Bersihkan GPU memory
gc.collect()
torch.cuda.empty_cache()

# Gunakan seq_length lebih rendah untuk T4
exp_max_seq_length = 1024  # T4-friendly (turun dari 2048)

print(f"VRAM before model load: {torch.cuda.memory_allocated()/1024**3:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

# Load model ulang untuk eksperimen 1
model_exp1, tokenizer_exp1 = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=exp_max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Setup tokenizer ChatML
tokenizer_exp1 = get_chat_template(
    tokenizer_exp1,
    chat_template="chatml",
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

# Apply LoRA — Experiment 1 hyperparameters
model_exp1 = FastLanguageModel.get_peft_model(
    model_exp1,
    r=16,  # LoRA rank — Experiment 1
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model Experiment 1 siap.")
print(f"LoRA rank (r): 16, lora_alpha: 16, max_seq_length: {exp_max_seq_length}")

In [ ]:
# Training Arguments — Experiment 1 (T4 Optimized)
training_args_exp1 = TrainingArguments(
    output_dir="./outputs/exp1",
    per_device_train_batch_size=1,        # T4: turun dari 4 → 1
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,        # T4: naik dari 4 → 8 (efektif batch=8)
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_steps=800,
    
    # [Skilled] Evaluation strategy
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    
    # [Skilled] Logging
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,
    
    # Optimization
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Precision — T4 no BF16, paksa FP16
    fp16=True,
    bf16=False,
    
    # Checkpointing
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Reporting
    report_to="none",
    save_total_limit=2,
    
    seed=42,
)

print("TrainingArguments Experiment 1 (T4 Optimized):")
print(f"  learning_rate: {training_args_exp1.learning_rate}")
print(f"  batch_size: {training_args_exp1.per_device_train_batch_size}")
print(f"  gradient_accumulation_steps: {training_args_exp1.gradient_accumulation_steps}")
print(f"  max_steps: {training_args_exp1.max_steps}")
print(f"  eval_strategy: {training_args_exp1.eval_strategy}")
print(f"  fp16: {training_args_exp1.fp16}, bf16: {training_args_exp1.bf16}")

In [ ]:
# SFTTrainer — Experiment 1
trainer_exp1 = SFTTrainer(
    model=model_exp1,
    tokenizer=tokenizer_exp1,
    args=training_args_exp1,
    train_dataset=train_dataset_mapped,
    eval_dataset=eval_dataset_mapped,  # [Skilled] Validation dataset
    dataset_text_field="text",
    max_seq_length=exp_max_seq_length,
    dataset_num_proc=2,
    packing=False,
)

print("SFTTrainer Experiment 1 siap.")
print(f"Training samples: {len(train_dataset_mapped)}")
print(f"Eval samples: {len(eval_dataset_mapped)}")
print(f"Effective batch size: {training_args_exp1.per_device_train_batch_size} × {training_args_exp1.gradient_accumulation_steps} = {training_args_exp1.per_device_train_batch_size * training_args_exp1.gradient_accumulation_steps}")

In [ ]:
# Jalankan training — Experiment 1
print("=" * 60)
print("MEMULAI TRAINING — EXPERIMENT 1")
print(f"  Model: {model_name}")
print(f"  LoRA r=16, lora_alpha=16")
print(f"  lr=2e-4, batch=1, accum=8, max_steps=800, seq_len={exp_max_seq_length}")
print(f"  FP16: True (T4 optimized)")
print("=" * 60)

trainer_exp1.train()

# Simpan log training
log_history_exp1 = trainer_exp1.state.log_history
print(f"\nTraining Experiment 1 selesai. Total log entries: {len(log_history_exp1)}")

## 9. Eksperimen 2 — Alternative Hyperparameters (T4 Optimized)

| Parameter | Value | Perubahan dari Exp 1 |
|-----------|-------|---------------------|
| learning_rate | 5e-5 | Lebih rendah (4x) |
| per_device_train_batch_size | 1 | Sama (T4 limit) |
| lora_r | 16 | Sama (T4 limit) |
| lora_alpha | 32 | Lebih besar (2x) |
| lora_dropout | 0.05 | Ada regularisasi |
| gradient_accumulation_steps | 8 | Sama |
| max_steps | 800 | Sama |
| max_seq_length | 1024 | Sama |
| lr_scheduler | cosine_with_restarts | Berbeda |
| max_grad_norm | 0.5 | Lebih ketat |
| eval_strategy | steps | Sama |
| eval_steps | 100 | Sama |
| logging_steps | 50 | Sama |

**Rasional**: lr lebih rendah + cosine_with_restarts + gradient clipping ketat → lebih stabil. lora_alpha lebih besar untuk coverage lebih baik tanpa naikkan VRAM.

In [ ]:
# Hapus model experiment 1 dari memory — PENTING untuk hemat VRAM
import gc
import shutil

# Hapus Unsloth compiled cache untuk bebas VRAM
cache_dir = "./unsloth_compiled_cache"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Unsloth cache cleared: {cache_dir}")

del model_exp1
del trainer_exp1
gc.collect()
torch.cuda.empty_cache()

# Verifikasi VRAM bersih
vram_used = torch.cuda.memory_allocated() / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Memory cleared — VRAM: {vram_used:.1f} GB / {vram_total:.1f} GB")

In [ ]:
# Load model ulang untuk eksperimen 2
print(f"VRAM before model load: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

model_exp2, tokenizer_exp2 = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=exp_max_seq_length,  # Gunakan 1024 (T4-friendly)
    dtype=None,
    load_in_4bit=True,
)

tokenizer_exp2 = get_chat_template(
    tokenizer_exp2,
    chat_template="chatml",
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

# Apply LoRA — Experiment 2: lora_alpha lebih besar, r tetap, gradient clipping ketat
model_exp2 = FastLanguageModel.get_peft_model(
    model_exp2,
    r=16,                  # Sama dengan Exp 1 (T4 constraint)
    lora_alpha=32,         # 2x Exp 1 — scaling factor lebih besar
    lora_dropout=0.05,     # Regularisasi kecil
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model Experiment 2 siap.")
print(f"LoRA rank (r): 16, lora_alpha: 32, lora_dropout: 0.05, max_seq_length: {exp_max_seq_length}")

In [ ]:
# Training Arguments — Experiment 2 (T4 Optimized, ultra-low VRAM)
training_args_exp2 = TrainingArguments(
    output_dir="./outputs/exp2",
    per_device_train_batch_size=1,        # T4 minimum
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,        # Efektif batch = 8
    learning_rate=5e-5,                   # 4x lebih rendah dari Exp 1
    lr_scheduler_type="cosine_with_restarts",
    warmup_ratio=0.05,
    max_steps=800,
    
    # [Skilled] Evaluation strategy
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    
    # [Skilled] Logging
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,
    
    # Optimization — ultra conservative
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.5,                    # Lebih ketat dari Exp 1 (1.0)
    
    # Precision — T4 no BF16
    fp16=True,
    bf16=False,
    
    # Checkpointing — minimal
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Reporting
    report_to="none",
    save_total_limit=1,                   # Hanya simpan best model
    
    # Memory saving
    dataloader_num_workers=0,             # No extra workers
    ddp_find_unused_parameters=False,
    
    seed=123,
)

print("TrainingArguments Experiment 2 (T4 Ultra-Low VRAM):")
print(f"  learning_rate: {training_args_exp2.learning_rate}")
print(f"  batch_size: {training_args_exp2.per_device_train_batch_size}")
print(f"  gradient_accumulation_steps: {training_args_exp2.gradient_accumulation_steps}")
print(f"  effective_batch: {training_args_exp2.per_device_train_batch_size * training_args_exp2.gradient_accumulation_steps}")
print(f"  lr_scheduler: {training_args_exp2.lr_scheduler_type}")
print(f"  max_grad_norm: {training_args_exp2.max_grad_norm}")
print(f"  warmup_ratio: {training_args_exp2.warmup_ratio}")
print(f"  max_steps: {training_args_exp2.max_steps}")

In [ ]:
# SFTTrainer — Experiment 2
trainer_exp2 = SFTTrainer(
    model=model_exp2,
    tokenizer=tokenizer_exp2,
    args=training_args_exp2,
    train_dataset=train_dataset_mapped,
    eval_dataset=eval_dataset_mapped,
    dataset_text_field="text",
    max_seq_length=exp_max_seq_length,
    dataset_num_proc=2,
    packing=False,
)

print("SFTTrainer Experiment 2 siap.")
print(f"Effective batch size: {training_args_exp2.per_device_train_batch_size} × {training_args_exp2.gradient_accumulation_steps} = {training_args_exp2.per_device_train_batch_size * training_args_exp2.gradient_accumulation_steps}")

In [ ]:
# Jalankan training — Experiment 2
print("=" * 60)
print("MEMULAI TRAINING — EXPERIMENT 2")
print(f"  Model: {model_name}")
print(f"  LoRA r=16, lora_alpha=32, dropout=0.05")
print(f"  lr=5e-5, batch=1, accum=8, max_steps=800, seq_len={exp_max_seq_length}")
print(f"  max_grad_norm=0.5, warmup=5%, FP16=True")
print("=" * 60)

trainer_exp2.train()

# Simpan log training
log_history_exp2 = trainer_exp2.state.log_history
print(f"\nTraining Experiment 2 selesai. Total log entries: {len(log_history_exp2)}")

## 10. Loss Curve Comparison

Membandingkan kurva training loss dan evaluation loss dari kedua eksperimen untuk menentukan hyperparameter mana yang menghasilkan training paling optimal tanpa overfitting.

In [ ]:
# Ekstrak training dan evaluation loss
def extract_losses(log_history):
    """Ekstrak training loss dan eval loss dari log history."""
    train_steps, train_losses = [], []
    eval_steps, eval_losses = [], []
    
    for entry in log_history:
        if "loss" in entry and "eval_loss" not in entry:
            train_steps.append(entry.get("step", 0))
            train_losses.append(entry["loss"])
        if "eval_loss" in entry:
            eval_steps.append(entry.get("step", 0))
            eval_losses.append(entry["eval_loss"])
    
    return train_steps, train_losses, eval_steps, eval_losses

t1_steps, t1_losses, e1_steps, e1_losses = extract_losses(log_history_exp1)
t2_steps, t2_losses, e2_steps, e2_losses = extract_losses(log_history_exp2)

print(f"Experiment 1 — Training loss points: {len(t1_losses)}, Eval loss points: {len(e1_losses)}")
print(f"Experiment 2 — Training loss points: {len(t2_losses)}, Eval loss points: {len(e2_losses)}")

In [ ]:
ax2.plot(e2_steps, e2_losses, label="Exp 2 (lr=5e-5, alpha=32)", color="#FF5722", 

In [ ]:
# Analisis: Pilih model terbaik berdasarkan eval loss terendah
min_e1 = min(e1_losses) if e1_losses else float("inf")
min_e2 = min(e2_losses) if e2_losses else float("inf")

print("=" * 60)
print("ANALISIS PERBANDINGAN")
print("=" * 60)
print(f"Experiment 1 (lr=2e-4, r=16, lora_alpha=16):")
print(f"  Training loss terakhir: {t1_losses[-1]:.4f}" if t1_losses else "  N/A")
print(f"  Eval loss terendah:    {min_e1:.4f}" if e1_losses else "  N/A")
print()
print(f"Experiment 2 (lr=5e-5, r=16, lora_alpha=32, dropout=0.05):")
print(f"  Training loss terakhir: {t2_losses[-1]:.4f}" if t2_losses else "  N/A")
print(f"  Eval loss terendah:    {min_e2:.4f}" if e2_losses else "  N/A")
print()

# Tentukan best model
if min_e1 < min_e2:
    print("KESIMPULAN: Experiment 1 menghasilkan eval loss lebih rendah.")
    print("Menggunakan model dari Experiment 1 untuk final merge.")
    BEST_MODEL = model_exp1
    BEST_TOKENIZER = tokenizer_exp1
else:
    print("KESIMPULAN: Experiment 2 menghasilkan eval loss lebih rendah.")
    print("Menggunakan model dari Experiment 2 untuk final merge.")
    BEST_MODEL = model_exp2
    BEST_TOKENIZER = tokenizer_exp2

## 11. Merge LoRA & Simpan Model

Merge adapters LoRA ke base model dalam format **merged_16bit** (FP16) lalu push ke HuggingFace Hub.

In [ ]:
# Merge LoRA adapters ke model dan simpan dalam 16-bit
# Gunakan merged_16bit = True sesuai ketentuan submission

print("Menyimpan model hasil fine-tuning (merged_16bit)...")

# Nama model untuk HuggingFace Hub
HF_USERNAME = "RayhanLup1n"
MODEL_NAME = "qwen2.5-7b-indonesian-legal-finetuned"

# Merge & save locally dalam 16-bit
BEST_MODEL.save_pretrained_merged(
    f"./{MODEL_NAME}",
    BEST_TOKENIZER,
    save_method="merged_16bit",  # [WAJIB] merged dalam 16-bit
)

print(f"Model disimpan di: ./{MODEL_NAME}/")

In [ ]:
# Push model ke HuggingFace Hub
print(f"Mengunggah model ke HuggingFace Hub: {HF_USERNAME}/{MODEL_NAME}")

# Push merged model (cara 1: via push_to_hub)
BEST_MODEL.push_to_hub_merged(
    f"{HF_USERNAME}/{MODEL_NAME}",
    BEST_TOKENIZER,
    save_method="merged_16bit",  # [WAJIB] merged dalam 16-bit
    token=True,  # Gunakan token dari huggingface-cli login
)

print(f"Model berhasil diunggah: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

## 12. Simpan Link HuggingFace

In [ ]:
# Simpan link HuggingFace ke file txt
hf_url = f"https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}"

with open("link_huggingface.txt", "w", encoding="utf-8") as f:
    f.write(hf_url)

print(f"Link disimpan ke link_huggingface.txt")
print(f"URL: {hf_url}")

- [x] eval_strategy="steps" + logging